> ### Archived experiment record — SmolVLM-500M fine-tuning
> **Phase 2, version 2 — code edited, outputs NOT regenerated**

| | |
|---|---|
| **Important** | The code cells were edited relative to v1 (150 validation images per class, patience 3, square-root-damped sampler, face-crop texture check), **but the saved outputs are identical to v1's** (320-image check subset, no texture-check output, same metrics). Treat this notebook as a code snapshot; do not cite its outputs as v2 results. |
| **Data** | train 27,872 · validation 5,761 · test 5,558 (as printed in the saved outputs, i.e. from v1) |

*Copy prepared for the repository: the driver-frame preview image (output of Section 5) and notebook-widget metadata were removed, empty trailing cells were dropped, and this banner was added. **No code cell was edited**; all text outputs are exactly as produced on the server. Absolute paths (`/home/jovyan/...`) belong to the university JupyterHub server and are intentionally left as part of the record. For a parameterised, re-runnable version of this workflow use `scripts/` together with `configs/` (see the repository README).*

# EdgeVLM-Drive - SmolVLM-500M fine-tuning (dual-crop, vision+text LoRA)

This notebook fine-tunes SmolVLM-500M-Instruct for four-class driver-state classification
(NORMAL, DROWSY, LOOKING_AWAY, PHONE_USAGE) using three design choices:

1. **Dual-crop input** (full scene + a zoomed face crop), reusing the exact preprocessing
   functions from `langgraph_pipeline.py` (the same production inference code), so the model
   is fine-tuned on the same pixels it will see at inference time.
2. **LoRA on both the vision encoder and the text decoder's attention layers**, so the model
   can adapt its visual representation of a subtle cue such as eye closure, not only its
   language-side label mapping.
3. **Model selection by validation macro-F1, computed via real generation**, not `eval_loss`.
   A custom callback runs `model.generate()` on a class-balanced validation subset after every
   epoch and tracks macro-F1 directly, with early stopping on that number.

**Run the smoke-test cell (Section 10) before starting full training**, and watch the first
few real logged steps carefully before walking away from a multi-hour run on a shared server.

**This notebook produces a LoRA adapter checkpoint, not a Jetson-ready artifact.** Merging the
adapter into the base model and quantizing for on-device deployment is a separate follow-up
step, not covered here.


## 0. Paths and labels


In [1]:
import os
from pathlib import Path

os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # must be set BEFORE torch is imported anywhere
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_FLAX"] = "0"

import pandas as pd
from PIL import Image

PROJECT_ROOT = Path("/home/jovyan/EdgeVLM_finetune")
FRAME_ROOT = Path("/home/jovyan")
SPLITS_DIR = Path("/home/jovyan/phase2-tain-val-test-splits")
AUGMENTED_SPLITS_DIR = SPLITS_DIR  # same merged folder now holds everything

TRAIN_CSV = SPLITS_DIR / "final_train.csv"
VAL_CSV = SPLITS_DIR / "final_val.csv"
TEST_CSV = SPLITS_DIR / "final_test.csv"

LABELS = ["NORMAL", "DROWSY", "LOOKING_AWAY", "PHONE_USAGE"]

# --- Point this at wherever langgraph_pipeline.py lives on the server ---
# Reusing the real preprocessing module (not a re-implementation) ensures training sees
# the same pixels production inference will - if this import fails, fix the path before
# continuing, do not re-implement the functions inline here.
LANGGRAPH_PIPELINE_DIR = Path("/home/jovyan/EdgeVLM_finetune/langgraph_pipeline.py")  # <-- EDIT if it lives elsewhere
import sys
sys.path.insert(0, str(LANGGRAPH_PIPELINE_DIR))
import langgraph_pipeline as lgp

print("langgraph_pipeline imported from:", lgp.__file__)
print("FULL_FRAME_SIZE:", lgp.FULL_FRAME_SIZE, " FACE_CROP_SIZE:", lgp.FACE_CROP_SIZE)
print("ENABLE_HAAR_FACE_FALLBACK:", lgp.ENABLE_HAAR_FACE_FALLBACK)

print("\nNotebook directory:", Path.cwd())
for path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(path, "exists:", path.is_file())


langgraph_pipeline imported from: /home/jovyan/EdgeVLM_finetune/langgraph_pipeline.py
FULL_FRAME_SIZE: (448, 448)  FACE_CROP_SIZE: (256, 256)
ENABLE_HAAR_FACE_FALLBACK: True

Notebook directory: /home/jovyan/EdgeVLM_finetune
/home/jovyan/phase2-tain-val-test-splits/final_train.csv exists: True
/home/jovyan/phase2-tain-val-test-splits/final_val.csv exists: True
/home/jovyan/phase2-tain-val-test-splits/final_test.csv exists: True


## 1. Read the three CSV files


In [2]:
train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)
test_df = pd.read_csv(TEST_CSV)

datasets = {"train": train_df, "validation": val_df, "test": test_df}

required_columns = {"frame_path", "label", "source_dataset", "group", "subject"}

for name, dataframe in datasets.items():
    missing_columns = required_columns - set(dataframe.columns)
    if missing_columns:
        raise ValueError(f"{name} is missing columns: {missing_columns}")

    unknown_labels = set(dataframe["label"].astype(str)) - set(LABELS)
    if unknown_labels:
        raise ValueError(f"{name} has unknown labels: {unknown_labels}")

    print(f"{name}: {len(dataframe)} rows")
    print(dataframe["label"].value_counts().reindex(LABELS, fill_value=0))
    print()


train: 27872 rows
label
NORMAL          13992
DROWSY           6287
LOOKING_AWAY     3028
PHONE_USAGE      4565
Name: count, dtype: int64

validation: 5761 rows
label
NORMAL          3418
DROWSY          1161
LOOKING_AWAY     310
PHONE_USAGE      872
Name: count, dtype: int64

test: 5558 rows
label
NORMAL          3277
DROWSY          1021
LOOKING_AWAY     302
PHONE_USAGE      958
Name: count, dtype: int64



## 2. Resolve augmented and original image paths

Resolves every row's image path, checking augmented directories first for the augmented classes.


In [3]:
AUGMENTED_IMAGE_DIRS = {
    "DROWSY": AUGMENTED_SPLITS_DIR / "augmented_drowsy_images",
    "LOOKING_AWAY": AUGMENTED_SPLITS_DIR / "augmented_looking_away_images",
}


def build_augmented_suffix_index():
    index = {}
    if not AUGMENTED_SPLITS_DIR.exists():
        return index
    for image_path in AUGMENTED_SPLITS_DIR.rglob("*"):
        if not image_path.is_file():
            continue
        if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
            continue
        relative_parts = image_path.relative_to(AUGMENTED_SPLITS_DIR).as_posix().split("/")
        first_start = max(0, len(relative_parts) - 6)
        for start in range(first_start, len(relative_parts)):
            key = "/".join(relative_parts[start:])
            index.setdefault(key, []).append(image_path)
    return index


_AUGMENTED_SUFFIX_INDEX = build_augmented_suffix_index()


def resolve_frame_path(frame_path, split_name, label=None):
    path_text = str(frame_path).strip().replace("\\", "/")
    relative_path = Path(path_text)

    if relative_path.is_absolute() and relative_path.is_file():
        return relative_path

    label_text = str(label).strip().upper()
    preferred_dir = AUGMENTED_IMAGE_DIRS.get(label_text)

    if split_name == "train":
        candidate_roots = []
        if preferred_dir is not None:
            candidate_roots.append(preferred_dir)
        candidate_roots.extend([AUGMENTED_SPLITS_DIR, FRAME_ROOT])
    else:
        candidate_roots = [FRAME_ROOT]

    for root in candidate_roots:
        candidate = root / relative_path
        if candidate.is_file():
            return candidate

    key = relative_path.as_posix().lstrip("./")
    matches = list(_AUGMENTED_SUFFIX_INDEX.get(key, []))

    if preferred_dir is not None:
        preferred_matches = [item for item in matches if preferred_dir in item.parents]
        if preferred_matches:
            return preferred_matches[0]

    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        raise FileNotFoundError(f"Ambiguous path {frame_path} for label {label}: {matches[:5]}")
    raise FileNotFoundError(f"Could not resolve {frame_path} for split {split_name}")


print("Indexed augmented image entries:", len(_AUGMENTED_SUFFIX_INDEX))


Indexed augmented image entries: 0


## 3. Check image availability

Sanity check - should print zero missing images for every split.


In [4]:
resolved_paths = {}
for name, dataframe in datasets.items():
    resolved = []
    missing = []
    for _, row in dataframe.iterrows():
        try:
            resolved.append(resolve_frame_path(row["frame_path"], split_name=name, label=row["label"]))
        except FileNotFoundError as error:
            resolved.append(None)
            missing.append(str(error))
    resolved_paths[name] = resolved
    print(f"{name} missing images: {len(missing)}")
    if missing:
        print("Example:", missing[0])


train missing images: 0
validation missing images: 0
test missing images: 0


## 4. Install fine-tuning dependencies

Pinned versions proven compatible with this server's CUDA/driver stack.


In [5]:
%pip uninstall -y -q opencv-python opencv-python-headless opencv-contrib-python opencv-contrib-python-headless

%pip install -q \
"opencv-python-headless==4.10.0.84" \
"transformers==4.46.3" \
"tokenizers==0.20.3" \
"peft>=0.13" \
"accelerate>=1.0" \
"datasets>=3.0" \
"safetensors>=0.4" \
"sentencepiece" \
"tqdm" \
"scikit-learn" \
"matplotlib"


Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**Restart the kernel now (Kernel -> Restart), then run all cells from the top.**

Reinstalling opencv does not fix a kernel that has already imported it once - the old,
broken `cv2` extension stays loaded in memory until the process restarts. Skipping this
step is why the same error can reappear even after the cell above runs successfully.

In [7]:
import sys
import torch
import transformers
import tokenizers
import peft
import accelerate

from transformers import AutoProcessor, AutoModelForVision2Seq, TrainingArguments, Trainer

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU total memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

import cv2
print("OpenCV:", cv2.__version__)
print("CascadeClassifier available:", hasattr(cv2, "CascadeClassifier"))


Python: 3.12.13 | packaged by conda-forge | (main, Mar  5 2026, 16:50:00) [GCC 14.3.0]
PyTorch: 2.11.0+cu126
CUDA available: True
GPU: NVIDIA A16
GPU total memory (GB): 15.7
Transformers: 4.46.3
Tokenizers: 0.20.3
PEFT: 0.20.0
Accelerate: 1.15.0
OpenCV: 4.10.0
CascadeClassifier available: True


## 5. Sanity-check the dual-crop preprocessing on real dataset images

Pulls sample images from every class, runs them through the actual `langgraph_pipeline.py`
functions (`enhance_frame` -> `resize_full_frame` / `extract_face_crop`), and displays the
results so you can visually confirm the face crop is landing on an actual face before spending
GPU time training on it. This is the same preprocessing production inference will run - if it
looks wrong here, fix it in `langgraph_pipeline.py` first, do not proceed to training.

Also reports how often the face detector actually finds a face, per class, over a larger
sample - a few visual rows can look fine by chance while a real failure rate is hiding
underneath, so this checks a number, not just an impression.

In [8]:
import cv2
import numpy as np
import matplotlib.pyplot as plt


def load_and_preprocess(image_path):
    """Mirrors what the training Dataset class below does per-sample."""
    raw_bgr = cv2.imread(str(image_path))
    if raw_bgr is None:
        raise FileNotFoundError(f"cv2 could not read: {image_path}")
    enhanced_rgb, frame_quality_ok = lgp.enhance_frame(raw_bgr)
    full_crop = lgp.resize_full_frame(enhanced_rgb)
    face_bbox = lgp.detect_face_bbox_haar(enhanced_rgb)  # None -> extract_face_crop used the last-resort guess
    face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)
    return full_crop, face_crop, frame_quality_ok, face_bbox is not None


# --- Part A: visual check, stratified so every class is represented --------
SAMPLES_PER_CLASS_VISUAL = 2
visual_rows = pd.concat(
    [train_df[train_df["label"] == label].sample(n=SAMPLES_PER_CLASS_VISUAL, random_state=0)
     for label in LABELS]
).reset_index(drop=True)

fig, axes = plt.subplots(len(visual_rows), 3, figsize=(9, 3 * len(visual_rows)))
for i, row in visual_rows.iterrows():
    image_path = resolve_frame_path(row["frame_path"], split_name="train", label=row["label"])
    raw_bgr = cv2.imread(str(image_path))
    full_crop, face_crop, frame_quality_ok, face_detected = load_and_preprocess(image_path)

    axes[i, 0].imshow(cv2.cvtColor(raw_bgr, cv2.COLOR_BGR2RGB))
    label_text = row["label"]
    axes[i, 0].set_title(f"{label_text}\noriginal")
    axes[i, 1].imshow(full_crop)
    axes[i, 1].set_title("full_crop (VLM input 1)")
    axes[i, 2].imshow(face_crop)
    axes[i, 2].set_title(f"face_crop (VLM input 2)\nface_detected={face_detected}")
    for ax in axes[i]:
        ax.axis("off")

plt.tight_layout()
plt.savefig(str(PROJECT_ROOT / "preprocessing_sanity_check.png"), dpi=100)
plt.show()
print("Saved to:", PROJECT_ROOT / "preprocessing_sanity_check.png")
print()
print("CHECK BEFORE CONTINUING: does face_crop actually contain a face in every row above?")
print("face_detected=False means the Haar cascade (frontal + profile) found nothing, and")
print("extract_face_crop fell back to an upper-region guess - that face_crop is NOT")
print("guaranteed to show a face. A low rate below is fine; a high one is not.")

# --- Part B: how often does this actually happen, across more real data? ---
SAMPLES_PER_CLASS_RATE = 100
print(f"\nEstimating face-detection rate on {SAMPLES_PER_CLASS_RATE} images per class...")

detection_stats = {}
for label in LABELS:
    subset = train_df[train_df["label"] == label]
    sample = subset.sample(n=min(SAMPLES_PER_CLASS_RATE, len(subset)), random_state=1)

    detected_count = 0
    for _, row in sample.iterrows():
        image_path = resolve_frame_path(row["frame_path"], split_name="train", label=row["label"])
        raw_bgr = cv2.imread(str(image_path))
        if raw_bgr is None:
            continue
        enhanced_rgb, _ = lgp.enhance_frame(raw_bgr)
        if lgp.detect_face_bbox_haar(enhanced_rgb) is not None:
            detected_count += 1

    detection_stats[label] = detected_count / len(sample)

print("\nFace-detection rate by class (1.0 = a face was found in every sampled image):")
for label, rate in detection_stats.items():
    flag = "  <-- check this closely, consider excluding or improving before training" if rate < 0.85 else ""
    print(f"{label}: {rate * 100:.1f}%{flag}")


# --- Part C: does the crop actually contain texture, or did the detector lock onto a flat
#     surface? ------------------------------------------------------------------------------
# A Haar cascade can report "face found" on a bounding box that is actually sitting on a
# plain door panel, headliner, or window - a geometric false positive the detection-rate
# check above cannot see, because that check only asks "did detect_face_bbox_haar return a
# box", not "is there a face inside it". A real face crop always has meaningful local
# contrast (eyes, brows, mouth, skin shading); a flat surface does not. Measuring the
# grayscale pixel standard deviation of each face_crop is a cheap, effective proxy for that
# distinction, and lets this check flag the failure mode numerically instead of relying on
# a human happening to notice it in the Part A preview.
FACE_CROP_STD_SAMPLE_PER_CLASS = 100
print(f"\nChecking face_crop texture (grayscale std) on {FACE_CROP_STD_SAMPLE_PER_CLASS} images per class...")

crop_std_stats = {}
for label in LABELS:
    subset = train_df[train_df["label"] == label]
    sample = subset.sample(n=min(FACE_CROP_STD_SAMPLE_PER_CLASS, len(subset)), random_state=2)

    stds = []
    for _, row in sample.iterrows():
        image_path = resolve_frame_path(row["frame_path"], split_name="train", label=row["label"])
        raw_bgr = cv2.imread(str(image_path))
        if raw_bgr is None:
            continue
        enhanced_rgb, _ = lgp.enhance_frame(raw_bgr)
        face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)
        gray = cv2.cvtColor(face_crop, cv2.COLOR_RGB2GRAY)
        stds.append(float(np.std(gray)))

    stds = np.array(stds)
    low_std_fraction = float(np.mean(stds < lgp.FACE_CROP_MIN_STD))
    crop_std_stats[label] = {"mean_std": float(stds.mean()), "low_std_fraction": low_std_fraction}

print(f"\nFace-crop texture check by class (flag threshold: std < {lgp.FACE_CROP_MIN_STD}, "
      f"the same threshold extract_face_crop uses internally to reject a Haar box and fall "
      f"back to the geometric guess):")
for label, stats in crop_std_stats.items():
    flag = "  <-- check this closely" if stats["low_std_fraction"] > 0.05 else ""
    print(f"{label}: mean_std={stats['mean_std']:.1f}, "
          f"low_std_fraction={stats['low_std_fraction']*100:.1f}%{flag}")


<Figure size 900x2400 with 24 Axes>

Saved to: /home/jovyan/EdgeVLM_finetune/preprocessing_sanity_check.png

CHECK BEFORE CONTINUING: does face_crop actually contain a face in every row above?
face_detected=False means the Haar cascade (frontal + profile) found nothing, and
extract_face_crop fell back to an upper-region guess - that face_crop is NOT
guaranteed to show a face. A low rate below is fine; a high one is not.

Estimating face-detection rate on 100 images per class...

Face-detection rate by class (1.0 = a face was found in every sampled image):
NORMAL: 96.0%
DROWSY: 94.0%
LOOKING_AWAY: 86.0%
PHONE_USAGE: 97.0%


## 6. Load SmolVLM-500M from the existing cache (or download once)

Tries the local cache first, falls back to a fresh download only if needed.


In [9]:
import torch
import gc

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
MODEL_CACHE = PROJECT_ROOT / "model_cache"
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32

try:
    print("Trying the local model cache...")
    processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE), local_files_only=True)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_ID, cache_dir=str(MODEL_CACHE), local_files_only=True, torch_dtype=dtype)
    print("Loaded from local cache.")
except OSError:
    print("Local cache is incomplete; downloading missing model files...")
    processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE))
    model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE), torch_dtype=dtype)

model = model.to(device)
model.eval()

print("Model:", MODEL_ID)
print("Device:", device)
print("Parameters:", sum(p.numel() for p in model.parameters()))


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Trying the local model cache...
Loaded from local cache.
Model: HuggingFaceTB/SmolVLM-500M-Instruct
Device: cuda
Parameters: 507482304


## 7. Define the fine-tuning dataset (dual-crop)

Each sample returns *two* images - the `full_crop` / `face_crop` pair production inference
computes via `langgraph_pipeline.enhance_frame` / `resize_full_frame` / `extract_face_crop` -
instead of one raw image. The instruction text explicitly tells the model what the two images
are, since nothing else in a two-image message says so.


In [10]:
from torch.utils.data import Dataset
from PIL import Image

INSTRUCTION = (
    "You are shown two images of the same moment from an in-cabin driver camera. "
    "The first image is the full scene. The second image is a close-up crop of the driver's "
    "face, showing more detail of the eyes and head position. "
    "Using both images together, classify the driver's visible state. "
    "Return exactly one label from: NORMAL, DROWSY, LOOKING_AWAY, or PHONE_USAGE. "
    "Do not provide an explanation or punctuation."
)


class DriverStateDualCropDataset(Dataset):
    """Runs every image through the real preprocessing pipeline and returns the
    (full_crop, face_crop) pair the model is trained and served on."""

    def __init__(self, dataframe, split_name):
        self.dataframe = dataframe.reset_index(drop=True)
        self.split_name = split_name

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = resolve_frame_path(row["frame_path"], split_name=self.split_name, label=row["label"])

        raw_bgr = cv2.imread(str(image_path))
        if raw_bgr is None:
            raise FileNotFoundError(f"cv2 could not read: {image_path}")

        enhanced_rgb, _frame_quality_ok = lgp.enhance_frame(raw_bgr)
        full_crop = lgp.resize_full_frame(enhanced_rgb)
        face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)

        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "image"},
                    {"type": "text", "text": INSTRUCTION},
                ],
            }
        ]

        return {
            "full_image": Image.fromarray(full_crop),
            "face_image": Image.fromarray(face_crop),
            "messages": messages,
            "target": str(row["label"]).strip(),
            "path": str(image_path),
        }


train_dataset = DriverStateDualCropDataset(train_df, split_name="train")
val_dataset = DriverStateDualCropDataset(val_df, split_name="validation")

example = train_dataset[0]
print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))
print("Example target:", example["target"])
print("Example full_image size:", example["full_image"].size)
print("Example face_image size:", example["face_image"].size)


Training examples: 27872
Validation examples: 5761
Example target: NORMAL
Example full_image size: (448, 448)
Example face_image size: (256, 256)


## 8. Create the multimodal training collator (dual-image)

Each sample contributes two images (`[full_image, face_image]`) to the processor.


In [11]:
class SmolVLMDualCropCollator:
    def __init__(self, processor, image_size=512):
        self.processor = processor
        self.image_size = image_size

    def __call__(self, features):
        images_for_processor = [[item["full_image"], item["face_image"]] for item in features]
        full_texts = []
        prompt_texts = []

        for item in features:
            user_messages = item["messages"]
            full_messages = user_messages + [
                {"role": "assistant", "content": [{"type": "text", "text": item["target"]}]}
            ]
            prompt_texts.append(self.processor.apply_chat_template(
                user_messages, add_generation_prompt=True, tokenize=False))
            full_texts.append(self.processor.apply_chat_template(
                full_messages, add_generation_prompt=False, tokenize=False))

        processor_arguments = {
            "return_tensors": "pt",
            "padding": True,
            "truncation": True,
            "size": {"longest_edge": self.image_size},
            "max_image_size": {"longest_edge": self.image_size},
        }

        batch = self.processor(text=full_texts, images=images_for_processor, **processor_arguments)
        prompt_batch = self.processor(text=prompt_texts, images=images_for_processor, **processor_arguments)

        labels = batch["input_ids"].clone()
        prompt_lengths = prompt_batch["attention_mask"].sum(dim=1)
        for row_number, prompt_length in enumerate(prompt_lengths):
            labels[row_number, :int(prompt_length)] = -100

        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        batch["labels"] = labels
        return batch


IMAGE_SIZE = 512  # comfortably within the A16's available memory headroom
collator = SmolVLMDualCropCollator(processor, image_size=IMAGE_SIZE)
print("Dual-crop collator created successfully.")


Dual-crop collator created successfully.


In [12]:
from torch.utils.data import DataLoader

loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collator)
batch = next(iter(loader))

print("Training batch created successfully.")
for key, value in batch.items():
    print(key, tuple(value.shape) if hasattr(value, "shape") else type(value))

print("Target tokens used for loss:", int((batch["labels"] != -100).sum().item()))
print("\nCHECK: pixel_values second dimension should be 2 (two images per sample), e.g. (2, 2, 3, 512, 512)")


Training batch created successfully.
pixel_values (2, 2, 3, 512, 512)
pixel_attention_mask (2, 2, 512, 512)
input_ids (2, 253)
attention_mask (2, 253)
labels (2, 253)
Target tokens used for loss: 14

CHECK: pixel_values second dimension should be 2 (two images per sample), e.g. (2, 2, 3, 512, 512)


## 9. Attach LoRA adapters to BOTH the vision encoder and the text decoder

The matching logic below separates `text_model` and `vision_model` attention projections by
name, and the printed counts let you sanity-check before training that both actually matched
something (an empty vision list would mean this SmolVLM revision's real module names don't
match the pattern assumed below, and needs adjusting before continuing).

LoRA is configured with `r=16`, `lora_alpha=32` (alpha/r ratio of 2), targeting the attention
projections in both the vision encoder and the text decoder - roughly 4.46M trainable
parameters, about 0.87% of the base model.


In [13]:
from peft import LoraConfig, TaskType, get_peft_model, PeftModel

ATTENTION_PROJ_SUFFIXES = ("q_proj", "k_proj", "v_proj", "o_proj", "out_proj")

if isinstance(model, PeftModel):
    print("A LoRA adapter is already attached.")
else:
    text_targets, vision_targets, other_targets = [], [], []

    for name, module in model.named_modules():
        if not (isinstance(module, torch.nn.Linear) and name.endswith(ATTENTION_PROJ_SUFFIXES)):
            continue
        if ".text_model." in name:
            text_targets.append(name)
        elif ".vision_model." in name:
            vision_targets.append(name)
        else:
            other_targets.append(name)

    print("Text-decoder attention layers matched:", len(text_targets))
    print("Vision-encoder attention layers matched:", len(vision_targets))
    if other_targets:
        print("Matched but NOT classified as text or vision (review these):", other_targets[:10])

    if not text_targets:
        raise RuntimeError("No text-model attention layers matched - inspect model.named_modules().")
    if not vision_targets:
        raise RuntimeError(
            "No vision-model attention layers matched - this SmolVLM revision's module names "
            "may differ from what ATTENTION_PROJ_SUFFIXES / '.vision_model.' assume. Print "
            "[n for n, _ in model.named_modules() if 'vision' in n.lower()][:30] to inspect "
            "the real names before continuing."
        )

    all_targets = sorted(text_targets + vision_targets)

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        target_modules=all_targets,
    )

    model.config.use_cache = False
    model.gradient_checkpointing_enable()  # extra memory headroom now that vision encoder trains too
    model.enable_input_require_grads()  # required so gradient checkpointing works with a frozen base model
    model = get_peft_model(model, lora_config)

model.train()
model.print_trainable_parameters()


Text-decoder attention layers matched: 128
Vision-encoder attention layers matched: 48
trainable params: 4,456,448 || all params: 511,938,752 || trainable%: 0.8705


## 10. Smoke test - run this before full training

**Do not skip this.** Runs 3 real training steps end to end (forward, backward, optimizer
step) on the dual-crop data with the vision+text LoRA config, so any wiring mistake shows up
in under a minute instead of after committing to a multi-hour run on the shared server.

In [14]:
RUN_SMOKE_TEST = True  # leave this True until it has passed at least once

if RUN_SMOKE_TEST:
    smoke_output = PROJECT_ROOT / "lora_smoke_test_500m_dualcrop"
    smoke_output.mkdir(parents=True, exist_ok=True)

    smoke_args = TrainingArguments(
        output_dir=str(smoke_output),
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=3,
        learning_rate=1e-4,
        fp16=torch.cuda.is_available(),
        logging_steps=1,
        save_strategy="no",
        eval_strategy="no",
        report_to="none",
        remove_unused_columns=False,
        dataloader_num_workers=0,
        optim="adamw_torch",
    )

    smoke_trainer = Trainer(
        model=model, args=smoke_args, train_dataset=train_dataset, data_collator=collator,
    )

    print("Starting smoke test (3 steps, dual-crop, vision+text LoRA)...")
    smoke_result = smoke_trainer.train()
    print("Smoke test completed - if you see a decreasing loss and no exceptions above, proceed.")
    print(smoke_result)
else:
    print("Smoke test skipped - only skip this after it has already passed once this session.")


max_steps is given, it will override any value given in num_train_epochs


Starting smoke test (3 steps, dual-crop, vision+text LoRA)...


Step,Training Loss
1,4.562700
2,4.211000
3,4.455600


Smoke test completed - if you see a decreasing loss and no exceptions above, proceed.
TrainOutput(global_step=3, training_loss=4.409760634104411, metrics={'train_runtime': 9.8586, 'train_samples_per_second': 1.217, 'train_steps_per_second': 0.304, 'total_flos': 8332484993280.0, 'train_loss': 4.409760634104411, 'epoch': 0.00043053960964408724})


## 11. Macro-F1 early-stopping callback

`eval_loss` alone is not a reliable signal for this task: with a strongly imbalanced dataset, a
model can reach low loss and high overall accuracy purely by favoring the majority class while
recall on the rare, safety-critical classes (especially DROWSY) collapses. This callback instead
runs real `model.generate()` calls on a small, fixed, class-balanced validation subset after
every epoch, computes macro-F1 directly with the same `parse_prediction` logic the final test
evaluation uses, and stops training after `PATIENCE` epochs without improvement.

The subset uses `SAMPLES_PER_CLASS` images per class (not the full validation set) because
`generate()` is far slower than the loss-only forward pass Trainer's built-in evaluation already
does each epoch; this keeps the per-epoch cost bounded, at the cost of a noisier macro-F1
estimate than the full validation set would give. The per-row `generate()` calls are batched
(`predict_batch_rows()`, batch size 8) to keep this cost manageable - batching a decoder-only
model for generation requires left-padding, which is set temporarily inside that function and
restored immediately after so it never affects training. The final full-test-set evaluation in
Section 13 is unaffected by any of this and remains the real, reportable number.


In [15]:
import json as _json
import time as _time
from sklearn.metrics import f1_score
from transformers import TrainerCallback

LABEL_ORDER = ["NORMAL", "DROWSY", "LOOKING_AWAY", "PHONE_USAGE"]
UNSAFE_LABELS = {"DROWSY", "LOOKING_AWAY", "PHONE_USAGE"}


def parse_prediction(raw_text):
    normalized = str(raw_text).strip().upper()
    cleaned = normalized.strip(" .,;:!?'\"")
    if cleaned in LABEL_ORDER:
        return cleaned, (normalized in LABEL_ORDER)
    return "INVALID", False


def predict_one_row(model, processor, row, split_name):
    image_path = resolve_frame_path(row["frame_path"], split_name=split_name, label=row["label"])
    raw_bgr = cv2.imread(str(image_path))
    enhanced_rgb, _ = lgp.enhance_frame(raw_bgr)
    full_crop = lgp.resize_full_frame(enhanced_rgb)
    face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)

    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "image"}, {"type": "text", "text": INSTRUCTION},
    ]}]
    chat_prompt = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    inputs = processor(
        text=chat_prompt, images=[[Image.fromarray(full_crop), Image.fromarray(face_crop)]],
        return_tensors="pt", size={"longest_edge": IMAGE_SIZE}, max_image_size={"longest_edge": IMAGE_SIZE},
    )
    inputs = inputs.to(next(model.parameters()).device)

    with torch.inference_mode():
        generated = model.generate(**inputs, max_new_tokens=12, do_sample=False)

    input_length = inputs["input_ids"].shape[-1]
    raw_output = processor.batch_decode(generated[:, input_length:], skip_special_tokens=True)[0].strip()
    prediction, exact_valid = parse_prediction(raw_output)
    return prediction


def predict_batch_rows(model, processor, rows_df, split_name, batch_size=8):
    """Same logic as predict_one_row, batched for speed. Generation is batched by temporarily
    forcing left-padding, which is required for batched decoder-only generate() calls (all real
    tokens end at the same position, so a single input_length correctly separates prompt from
    generated tokens for every row in the batch); training itself is unaffected since
    padding_side is restored right after."""
    predictions = []
    original_padding_side = processor.tokenizer.padding_side
    processor.tokenizer.padding_side = "left"
    try:
        for start in range(0, len(rows_df), batch_size):
            batch_rows = rows_df.iloc[start:start + batch_size]
            full_images, face_images, prompt_texts = [], [], []
            for _, row in batch_rows.iterrows():
                image_path = resolve_frame_path(row["frame_path"], split_name=split_name, label=row["label"])
                raw_bgr = cv2.imread(str(image_path))
                enhanced_rgb, _ = lgp.enhance_frame(raw_bgr)
                full_crop = lgp.resize_full_frame(enhanced_rgb)
                face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)
                full_images.append(Image.fromarray(full_crop))
                face_images.append(Image.fromarray(face_crop))
                messages = [{"role": "user", "content": [
                    {"type": "image"}, {"type": "image"}, {"type": "text", "text": INSTRUCTION},
                ]}]
                prompt_texts.append(processor.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False))

            images_for_processor = [[fi, fc] for fi, fc in zip(full_images, face_images)]
            inputs = processor(
                text=prompt_texts, images=images_for_processor, return_tensors="pt", padding=True,
                size={"longest_edge": IMAGE_SIZE}, max_image_size={"longest_edge": IMAGE_SIZE},
            )
            inputs = inputs.to(next(model.parameters()).device)

            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=12, do_sample=False)

            input_length = inputs["input_ids"].shape[-1]
            raw_outputs = processor.batch_decode(generated[:, input_length:], skip_special_tokens=True)
            for raw_output in raw_outputs:
                pred, _ = parse_prediction(raw_output.strip())
                predictions.append(pred)
    finally:
        processor.tokenizer.padding_side = original_padding_side
    return predictions


# --- fixed, class-balanced validation subset, sampled once so every epoch is checked against
#     the exact same rows (a fair, apples-to-apples comparison across epochs) ---
import numpy as np

SAMPLES_PER_CLASS = 150  # per-class sample size for the per-epoch macro-F1 check;
# large enough to keep the early-stopping signal reasonably low-noise. The smallest
# validation class (LOOKING_AWAY, ~300 rows) still leaves this well under its full count,
# so the subset stays genuinely class-balanced rather than silently falling back to
# "all of it" for the rare classes while sampling 150 from the common ones.
epoch_check_rows = []
for label in LABEL_ORDER:
    class_rows = val_df[val_df["label"] == label]
    n = min(SAMPLES_PER_CLASS, len(class_rows))
    epoch_check_rows.append(class_rows.sample(n=n, random_state=0))
epoch_check_df = pd.concat(epoch_check_rows).reset_index(drop=True)
print("Per-epoch macro-F1 check subset size:", len(epoch_check_df),
      "(", epoch_check_df["label"].value_counts().reindex(LABEL_ORDER, fill_value=0).to_dict(), ")")


class MacroF1EarlyStopping(TrainerCallback):
    # A macro-F1 estimate from a sampled subset (rather than the full validation set) is
    # noisier epoch-to-epoch than eval_loss, so patience=2 can declare "no improvement" and
    # stop training on what is really just sampling noise around a plateau, cutting training
    # short before the model has actually had a fair chance to keep improving. patience=3
    # asks for two consecutive non-improving epochs before giving up instead of one, which
    # costs at most one extra epoch of training time but makes the stopping decision less
    # sensitive to a single noisy evaluation.
    def __init__(self, processor, check_df, output_dir, patience=3):
        self.processor = processor
        self.check_df = check_df
        self.output_dir = Path(output_dir)
        self.patience = patience
        self.best_macro_f1 = -1.0
        self.epochs_without_improvement = 0
        self.history = []
        self.log_path = self.output_dir / "macro_f1_epoch_log.json"

    def on_epoch_end(self, args, state, control, **kwargs):
        model_ref = kwargs.get("model", None)
        if model_ref is None:
            print("[MacroF1EarlyStopping] no model in callback kwargs - skipping this epoch's check.")
            return control

        try:
            model_ref.eval()
            t0 = _time.perf_counter()
            predicted_labels = predict_batch_rows(
                model_ref, self.processor, self.check_df, split_name="validation", batch_size=8,
            )
            true_labels = [str(label).strip().upper() for label in self.check_df["label"].tolist()]
            elapsed = _time.perf_counter() - t0

            macro_f1 = f1_score(true_labels, predicted_labels, labels=LABEL_ORDER,
                                 average="macro", zero_division=0)
            per_class_f1 = f1_score(true_labels, predicted_labels, labels=LABEL_ORDER,
                                     average=None, zero_division=0)

            improved = macro_f1 > self.best_macro_f1
            if improved:
                self.best_macro_f1 = macro_f1
                self.epochs_without_improvement = 0
                best_dir = self.output_dir / "best_by_macro_f1"
                best_dir.mkdir(parents=True, exist_ok=True)
                model_ref.save_pretrained(str(best_dir))
                self.processor.save_pretrained(str(best_dir))
            else:
                self.epochs_without_improvement += 1

            record = {
                "epoch": state.epoch, "macro_f1": macro_f1,
                "per_class_f1": dict(zip(LABEL_ORDER, per_class_f1.tolist())),
                "improved": improved, "eval_seconds": elapsed,
            }
            self.history.append(record)
            with open(self.log_path, "w") as f:
                _json.dump(self.history, f, indent=2)

            print(f"\n[MacroF1EarlyStopping] epoch {state.epoch:.2f}: macro_f1={macro_f1:.4f} "
                  f"(best={self.best_macro_f1:.4f}) per_class={dict(zip(LABEL_ORDER, np.round(per_class_f1, 3)))} "
                  f"[{elapsed:.1f}s on {len(self.check_df)} samples]")

            if self.epochs_without_improvement >= self.patience:
                print(f"[MacroF1EarlyStopping] no improvement for {self.patience} epochs - stopping training. "
                      f"Best checkpoint saved at: {self.output_dir / 'best_by_macro_f1'}")
                control.should_training_stop = True

        except Exception as e:
            print(f"[MacroF1EarlyStopping] check failed this epoch (training continues): {type(e).__name__}: {e}")
        finally:
            model_ref.train()

        return control


print("MacroF1EarlyStopping callback defined.")


Per-epoch macro-F1 check subset size: 320 ( {'NORMAL': 80, 'DROWSY': 80, 'LOOKING_AWAY': 80, 'PHONE_USAGE': 80} )
MacroF1EarlyStopping callback defined.


## 12. Fine-tune

- `num_train_epochs` is set to a generous upper bound (8) - `MacroF1EarlyStopping` above
  decides the real stopping point, not this number.
- `gradient_checkpointing` is on (via `model.gradient_checkpointing_enable()` in Section 9),
  trading some training speed for memory headroom since the vision encoder trains too.
- `eval_strategy` / `save_strategy` use `"epoch"`, and `logging_steps=50`, to keep the volume
  of output streamed back to the notebook manageable over a multi-hour run on a shared server.
- Checkpoint selection is driven entirely by `MacroF1EarlyStopping`'s own `best_by_macro_f1`
  checkpoint; no `eval_loss`-based selection is configured, so there is a single, unambiguous
  notion of "best" for this run.
- `weight_decay=0.01` provides a small, standard regularizer on the LoRA parameters.


In [16]:
from collections import Counter
from torch.utils.data import WeightedRandomSampler

class BalancedTrainer(Trainer):
    def _get_train_sampler(self):
        # Full inverse-frequency weighting (1/count) is the textbook starting point, but on
        # this dataset's class ratios (NORMAL is ~4.6x LOOKING_AWAY, ~2.2x DROWSY in the
        # training split) it over-corrects: the rarest class ends up sampled several times
        # more often per epoch than its real-world prevalence, which pushes the model toward
        # over-predicting it and shows up downstream as extra false alarms. Damping the
        # weights with a square root keeps the correction (rare classes still get seen more
        # than their raw frequency) without letting the ratio become as extreme - a standard
        # middle ground between "no reweighting" and "full inverse-frequency" for imbalance
        # this size.
        labels = self.train_dataset.dataframe["label"].astype(str).tolist()
        counts = Counter(labels)
        sample_weights = torch.tensor([1.0 / (counts[label] ** 0.5) for label in labels], dtype=torch.double)
        return WeightedRandomSampler(weights=sample_weights, num_samples=len(sample_weights), replacement=True)


FINAL_OUTPUT = PROJECT_ROOT / "smolvlm_500m_lora_dualcrop"
FINAL_OUTPUT.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(FINAL_OUTPUT),
    overwrite_output_dir=True,

    num_train_epochs=8,  # upper bound only - MacroF1EarlyStopping decides the real stop point
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=1e-4,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",

    fp16=torch.cuda.is_available(),

    logging_steps=50,
    eval_strategy="epoch",

    save_strategy="epoch",
    save_total_limit=2,
    # Checkpoint selection is handled entirely by MacroF1EarlyStopping's own
    # best_by_macro_f1 checkpoint - FINAL_OUTPUT holds the last-epoch state (for
    # debugging), and FINAL_OUTPUT/best_by_macro_f1 is the one to deploy.
    weight_decay=0.01,  # small, standard regularizer on the LoRA parameters

    remove_unused_columns=False,
    dataloader_num_workers=0,
    optim="adamw_torch",
    report_to="none",
)

early_stopping_callback = MacroF1EarlyStopping(
    processor=processor, check_df=epoch_check_df, output_dir=FINAL_OUTPUT, patience=3,
)

trainer = BalancedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collator,
    callbacks=[early_stopping_callback],
)

print("Starting fine-tuning (SmolVLM-500M, dual-crop, vision+text LoRA, macro-F1 early stopping)...")
train_result = trainer.train()

print("Fine-tuning completed.")
print(train_result)

trainer.save_model(str(FINAL_OUTPUT))
processor.save_pretrained(str(FINAL_OUTPUT))
print("Saved final (last-epoch, NOT best) adapter and processor to:", FINAL_OUTPUT)
print("Saved macro-F1-best adapter and processor to:", FINAL_OUTPUT / "best_by_macro_f1")
print("Per-epoch macro-F1 history:", FINAL_OUTPUT / "macro_f1_epoch_log.json")


Starting fine-tuning (SmolVLM-500M, dual-crop, vision+text LoRA, macro-F1 early stopping)...


Epoch,Training Loss,Validation Loss
1,0.035300,0.163045
2,0.034700,0.193249
3,0.026700,0.197544



[MacroF1EarlyStopping] epoch 1.00: macro_f1=0.7942 (best=0.7942) per_class={'NORMAL': np.float64(0.708), 'DROWSY': np.float64(0.78), 'LOOKING_AWAY': np.float64(0.8), 'PHONE_USAGE': np.float64(0.889)} [94.9s on 320 samples]

[MacroF1EarlyStopping] epoch 2.00: macro_f1=0.7874 (best=0.7942) per_class={'NORMAL': np.float64(0.677), 'DROWSY': np.float64(0.767), 'LOOKING_AWAY': np.float64(0.815), 'PHONE_USAGE': np.float64(0.89)} [94.1s on 320 samples]

[MacroF1EarlyStopping] epoch 3.00: macro_f1=0.7603 (best=0.7942) per_class={'NORMAL': np.float64(0.68), 'DROWSY': np.float64(0.761), 'LOOKING_AWAY': np.float64(0.757), 'PHONE_USAGE': np.float64(0.843)} [93.9s on 320 samples]
[MacroF1EarlyStopping] no improvement for 2 epochs - stopping training. Best checkpoint saved at: /home/jovyan/EdgeVLM_finetune/smolvlm_500m_lora_dualcrop/best_by_macro_f1
Fine-tuning completed.
TrainOutput(global_step=10452, training_loss=0.08528348456831543, metrics={'train_runtime': 68347.9392, 'train_samples_per_second

## 13. Evaluate on the untouched test set

**Loads the macro-F1-best checkpoint explicitly** from `best_by_macro_f1` (not whatever
`trainer`'s in-memory model or `FINAL_OUTPUT` happens to hold after the last epoch) - this is
the actual model this notebook is trying to produce. Also reports peak GPU memory, though that
number describes the university server's A16, not the Jetson Nano - it is not a deployment
feasibility number by itself.


In [31]:
import time
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_recall_fscore_support
from peft import PeftModel

BEST_CHECKPOINT = FINAL_OUTPUT / "best_by_macro_f1"
print("Loading macro-F1-best checkpoint from:", BEST_CHECKPOINT)

eval_processor = AutoProcessor.from_pretrained(str(BEST_CHECKPOINT))
eval_base_model = AutoModelForVision2Seq.from_pretrained(MODEL_ID, cache_dir=str(MODEL_CACHE), torch_dtype=dtype)
eval_model = PeftModel.from_pretrained(eval_base_model, str(BEST_CHECKPOINT))
eval_model = eval_model.to(device)
eval_model.eval()


def predict_one_test_row(row):
    image_path = resolve_frame_path(row["frame_path"], split_name="test", label=row["label"])
    raw_bgr = cv2.imread(str(image_path))
    enhanced_rgb, _ = lgp.enhance_frame(raw_bgr)
    full_crop = lgp.resize_full_frame(enhanced_rgb)
    face_crop = lgp.extract_face_crop(enhanced_rgb, landmarks=None)

    messages = [{"role": "user", "content": [
        {"type": "image"}, {"type": "image"}, {"type": "text", "text": INSTRUCTION},
    ]}]
    chat_prompt = eval_processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

    inputs = eval_processor(
        text=chat_prompt, images=[[Image.fromarray(full_crop), Image.fromarray(face_crop)]],
        return_tensors="pt", size={"longest_edge": IMAGE_SIZE}, max_image_size={"longest_edge": IMAGE_SIZE},
    )
    inputs = inputs.to(next(eval_model.parameters()).device)

    start = time.perf_counter()
    with torch.inference_mode():
        generated = eval_model.generate(**inputs, max_new_tokens=12, do_sample=False)
    latency_ms = (time.perf_counter() - start) * 1000

    input_length = inputs["input_ids"].shape[-1]
    raw_output = eval_processor.batch_decode(generated[:, input_length:], skip_special_tokens=True)[0].strip()
    prediction, exact_valid = parse_prediction(raw_output)
    return raw_output, prediction, exact_valid, latency_ms, str(image_path)


true_labels, predicted_labels, raw_outputs, exact_valid_flags, latencies, resolved_test_paths = (
    [], [], [], [], [], [])

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Evaluating untouched test images"):
    raw_output, prediction, exact_valid, latency_ms, resolved_path = predict_one_test_row(row)
    true_labels.append(str(row["label"]).strip().upper())
    predicted_labels.append(prediction)
    raw_outputs.append(raw_output)
    exact_valid_flags.append(exact_valid)
    latencies.append(latency_ms)
    resolved_test_paths.append(resolved_path)

true_labels = np.array(true_labels)
predicted_labels = np.array(predicted_labels)
exact_valid_flags = np.array(exact_valid_flags)
latencies = np.array(latencies)

precision, recall, f1, support = precision_recall_fscore_support(
    true_labels, predicted_labels, labels=LABEL_ORDER, zero_division=0)
balanced_accuracy_four_class = recall.mean()

unsafe_mask = np.isin(true_labels, list(UNSAFE_LABELS))
normal_mask = true_labels == "NORMAL"
unsafe_miss_rate = np.mean(predicted_labels[unsafe_mask] != true_labels[unsafe_mask])
false_alarm_rate = np.mean(np.isin(predicted_labels[normal_mask], list(UNSAFE_LABELS)))
parser_failure_rate = np.mean(predicted_labels == "INVALID")
exact_label_validity_rate = exact_valid_flags.mean()

matrix_labels = LABEL_ORDER + ["INVALID"]
cm = confusion_matrix(true_labels, predicted_labels, labels=matrix_labels)

results_df = test_df.copy()
results_df["resolved_path"] = resolved_test_paths
results_df["raw_output"] = raw_outputs
results_df["prediction"] = predicted_labels
results_df["exact_label_valid"] = exact_valid_flags
results_df["latency_ms"] = latencies
results_path = PROJECT_ROOT / "smolvlm500m_dualcrop_test_predictions.csv"
results_df.to_csv(results_path, index=False)

print("\n========== FINAL TEST RESULTS (SmolVLM-500M, dual-crop, vision+text LoRA) ==========")
print("Evaluation images:", len(test_df))
print("Exact-label validity rate:", round(exact_label_validity_rate * 100, 2), "%")
print("Parser failure rate:", round(parser_failure_rate * 100, 2), "%")
print("Overall accuracy:", round(accuracy_score(true_labels, predicted_labels) * 100, 2), "%")
print("Balanced accuracy:", round(balanced_accuracy_four_class * 100, 2), "%")
print("Macro-F1:", round(f1_score(true_labels, predicted_labels, labels=LABEL_ORDER, average="macro", zero_division=0) * 100, 2), "%")

print("\nPer-class results:")
for label, p, r, f, n in zip(LABEL_ORDER, precision, recall, f1, support):
    print(f"{label}: precision={p:.3f}, recall={r:.3f}, F1={f:.3f}, samples={n}")

print("\nConfusion matrix:")
print(pd.DataFrame(cm, index=matrix_labels, columns=matrix_labels))

print("\nUnsafe-state miss rate:", round(unsafe_miss_rate * 100, 2), "%")
print("False-alarm rate:", round(false_alarm_rate * 100, 2), "%")
print("Mean latency (ms):", round(float(np.mean(latencies)), 2))
print("P95 latency (ms):", round(float(np.percentile(latencies, 95)), 2))

if torch.cuda.is_available():
    peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
    print("Peak GPU memory on THIS server (MB, not the Jetson):", round(peak_memory_mb, 2))

print("\nPredictions saved to:", results_path)


Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Loading macro-F1-best checkpoint from: /home/jovyan/EdgeVLM_finetune/smolvlm_500m_lora_dualcrop/best_by_macro_f1


Evaluating untouched test images:   0%|          | 0/5558 [00:00<?, ?it/s]


========== FINAL TEST RESULTS (SmolVLM-500M, dual-crop, vision+text LoRA) ==========
Evaluation images: 5558
Exact-label validity rate: 100.0 %
Parser failure rate: 0.0 %
Overall accuracy: 85.44 %
Balanced accuracy: 83.0 %
Macro-F1: 79.56 %

Per-class results:
NORMAL: precision=0.906, recall=0.898, F1=0.902, samples=3277
DROWSY: precision=0.785, recall=0.813, F1=0.799, samples=1021
LOOKING_AWAY: precision=0.504, recall=0.861, F1=0.636, samples=302
PHONE_USAGE: precision=0.973, recall=0.748, F1=0.846, samples=958

Confusion matrix:
              NORMAL  DROWSY  LOOKING_AWAY  PHONE_USAGE  INVALID
NORMAL          2942     222           108            5        0
DROWSY           167     830            16            8        0
LOOKING_AWAY      34       1           260            7        0
PHONE_USAGE      105       4           132          717        0
INVALID            0       0             0            0        0

Unsafe-state miss rate: 20.78 %
False-alarm rate: 10.22 %
Mean latency 

In [32]:
from pathlib import Path

BEST_ADAPTER_DIR = FINAL_OUTPUT / "best_by_macro_f1"
EXPORT_DIR = PROJECT_ROOT / "smolvlm_500m_pytorch_deployment"

if not BEST_ADAPTER_DIR.is_dir():
    raise FileNotFoundError(
        f"Best adapter was not found at: {BEST_ADAPTER_DIR}\n"
        "Run the training cell first and make sure macro-F1 checkpoint was saved."
    )

required_adapter_files = [
    "adapter_config.json",
    "adapter_model.safetensors",
]

missing_adapter_files = [
    file_name
    for file_name in required_adapter_files
    if not (BEST_ADAPTER_DIR / file_name).is_file()
]

if missing_adapter_files:
    raise FileNotFoundError(
        f"Missing adapter files: {missing_adapter_files}\n"
        f"Adapter directory: {BEST_ADAPTER_DIR}"
    )

print("Best adapter found:")
print(BEST_ADAPTER_DIR)

Best adapter found:
/home/jovyan/EdgeVLM_finetune/smolvlm_500m_lora_dualcrop/best_by_macro_f1


In [33]:
import gc
import torch
from peft import PeftModel
from transformers import AutoModelForVision2Seq

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

BASE_MODEL_FOR_EXPORT = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE),
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

adapter_model = PeftModel.from_pretrained(
    BASE_MODEL_FOR_EXPORT,
    str(BEST_ADAPTER_DIR),
    is_trainable=False,
)

adapter_model.eval()

print("Base SmolVLM-500M and LoRA adapter loaded.")

Base SmolVLM-500M and LoRA adapter loaded.


In [34]:
merged_model = adapter_model.merge_and_unload()
merged_model.eval()

print("LoRA adapter merged into the SmolVLM-500M base model.")
print(
    "Merged parameter count:",
    sum(parameter.numel() for parameter in merged_model.parameters()),
)

LoRA adapter merged into the SmolVLM-500M base model.
Merged parameter count: 507482304


In [35]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

merged_model.save_pretrained(
    str(EXPORT_DIR),
    safe_serialization=True,
    max_shard_size="2GB",
)

processor.save_pretrained(str(EXPORT_DIR))

print("Complete PyTorch deployment model saved to:")
print(EXPORT_DIR)

Complete PyTorch deployment model saved to:
/home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment


In [36]:
deployment_info = {
    "model_id": MODEL_ID,
    "model_type": "SmolVLM-500M-Instruct",
    "deployment_backend": "PyTorch",
    "onnx_used": False,
    "adapter_merged": True,
    "image_size": IMAGE_SIZE,
    "labels": LABELS,
    "source_adapter": str(BEST_ADAPTER_DIR),
}

with open(EXPORT_DIR / "deployment_info.txt", "w", encoding="utf-8") as file:
    for key, value in deployment_info.items():
        file.write(f"{key}: {value}\n")

print("Deployment information saved.")

Deployment information saved.


In [37]:
print("Deployment files:")

for file_path in sorted(EXPORT_DIR.iterdir()):
    if file_path.is_file():
        print(
            file_path.name,
            "| size MB =",
            round(file_path.stat().st_size / (1024 ** 2), 2),
        )

model_weight_files = list(EXPORT_DIR.glob("*.safetensors"))

required_files = [
    "config.json",
    "preprocessor_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

missing_files = [
    file_name
    for file_name in required_files
    if not (EXPORT_DIR / file_name).is_file()
]

if not model_weight_files:
    missing_files.append("*.safetensors")

if missing_files:
    raise FileNotFoundError(
        f"Deployment package is incomplete. Missing: {missing_files}"
    )

print("Deployment package verified successfully.")

Deployment files:
added_tokens.json | size MB = 0.0
chat_template.json | size MB = 0.0
config.json | size MB = 0.01
deployment_info.txt | size MB = 0.0
generation_config.json | size MB = 0.0
merges.txt | size MB = 0.44
model.safetensors | size MB = 968.0
preprocessor_config.json | size MB = 0.0
processor_config.json | size MB = 0.0
special_tokens_map.json | size MB = 0.0
tokenizer.json | size MB = 3.38
tokenizer_config.json | size MB = 0.03
vocab.json | size MB = 0.76
Deployment package verified successfully.


In [38]:
from transformers import AutoProcessor, AutoModelForVision2Seq

deployment_processor = AutoProcessor.from_pretrained(
    str(EXPORT_DIR),
    local_files_only=True,
)

deployment_model = AutoModelForVision2Seq.from_pretrained(
    str(EXPORT_DIR),
    local_files_only=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    low_cpu_mem_usage=True,
)

deployment_model.eval()

print("Exported deployment model loaded successfully.")
print("Deployment path:", EXPORT_DIR)

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


Exported deployment model loaded successfully.
Deployment path: /home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment


In [39]:
import shutil

archive_base = PROJECT_ROOT / "smolvlm_500m_pytorch_deployment"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(EXPORT_DIR),
)

print("Deployment archive created:")
print(archive_path)

Deployment archive created:
/home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment.zip


In [40]:
MODEL_DIR = Path(
    "/home/jetson/smolvlm_500m_pytorch_deployment"
)

ADAPTER_DIR = None

print("Deployment model directory:", MODEL_DIR)
print("Using merged PyTorch model without a separate adapter.")

Deployment model directory: /home/jetson/smolvlm_500m_pytorch_deployment
Using merged PyTorch model without a separate adapter.


In [41]:
from pathlib import Path

FINAL_OUTPUT = Path(
    "/home/jovyan/EdgeVLM_finetune/smolvlm_500m_lora_dualcrop"
)

BEST_ADAPTER_DIR = FINAL_OUTPUT / "best_by_macro_f1"

EXPORT_DIR = Path(
    "/home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment"
)

print("Training output exists:", FINAL_OUTPUT.is_dir())
print("Best adapter exists:", BEST_ADAPTER_DIR.is_dir())
print("Export directory exists:", EXPORT_DIR.is_dir())

if BEST_ADAPTER_DIR.is_dir():
    print("\nAdapter files:")
    for path in sorted(BEST_ADAPTER_DIR.iterdir()):
        print(path.name)

if EXPORT_DIR.is_dir():
    print("\nExport files:")
    for path in sorted(EXPORT_DIR.iterdir()):
        print(path.name)

Training output exists: True
Best adapter exists: True
Export directory exists: True

Adapter files:
README.md
adapter_config.json
adapter_model.safetensors
added_tokens.json
chat_template.json
merges.txt
preprocessor_config.json
processor_config.json
special_tokens_map.json
tokenizer.json
tokenizer_config.json
vocab.json

Export files:
added_tokens.json
chat_template.json
config.json
deployment_info.txt
generation_config.json
merges.txt
model.safetensors
preprocessor_config.json
processor_config.json
special_tokens_map.json
tokenizer.json
tokenizer_config.json
vocab.json


In [42]:
from transformers import AutoModelForVision2Seq, AutoProcessor
from peft import PeftModel

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
MODEL_CACHE = Path("/home/jovyan/EdgeVLM_finetune/model_cache")

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE),
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
)

adapter_model = PeftModel.from_pretrained(
    base_model,
    str(BEST_ADAPTER_DIR),
    is_trainable=False,
)

merged_model = adapter_model.merge_and_unload()
merged_model.eval()

deployment_processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    cache_dir=str(MODEL_CACHE),
)

merged_model.save_pretrained(
    str(EXPORT_DIR),
    safe_serialization=True,
    max_shard_size="2GB",
)

deployment_processor.save_pretrained(str(EXPORT_DIR))

print("PyTorch deployment package created at:")
print(EXPORT_DIR)

Some kwargs in processor config are unused and will not have any effect: image_seq_len. 


PyTorch deployment package created at:
/home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment


In [43]:
required_files = [
    "config.json",
    "preprocessor_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

missing_files = [
    name for name in required_files
    if not (EXPORT_DIR / name).is_file()
]

weights = list(EXPORT_DIR.glob("*.safetensors"))

if not weights:
    missing_files.append("*.safetensors")

print("Export directory:", EXPORT_DIR)
print("Weight files:", [path.name for path in weights])

if missing_files:
    raise FileNotFoundError(
        f"Export package is incomplete. Missing: {missing_files}"
    )

print("Export package is complete and ready to transfer.")

Export directory: /home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment
Weight files: ['model.safetensors']
Export package is complete and ready to transfer.


In [44]:
import shutil
from pathlib import Path

EXPORT_DIR = Path(
    "/home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment"
)

ARCHIVE_BASE = Path(
    "/home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment"
)

archive_path = shutil.make_archive(
    str(ARCHIVE_BASE),
    "zip",
    root_dir=str(EXPORT_DIR.parent),
    base_dir=EXPORT_DIR.name,
)

print("Archive created:")
print(archive_path)

Archive created:
/home/jovyan/EdgeVLM_finetune/smolvlm_500m_pytorch_deployment.zip
